## Clone the Repository

In [ ]:
# Clone the project from GitHub
!git clone https://github.com/HamzaGbada/medical-ai-test.git
%cd medical-ai-test

Cloning into 'medical-ai-test'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 237 (delta 84), reused 237 (delta 84), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 3.19 MiB | 44.15 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/medical-ai-test


## Install dependencies

In [ ]:
!pip install -q \
    torch torchvision \
    medmnist>=2.2.0 \
    numpy pandas matplotlib seaborn \
    scikit-learn tqdm pyyaml \
    Pillow

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")


PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4


## HuggingFace Login

In [ ]:
from huggingface_hub import login

# Option A — Interactive prompt (will ask for your token)
login()

# Option B — Paste your token directly (replace YOUR_TOKEN_HERE)
# login(token="hf_YOUR_TOKEN_HERE")

## Upload a pre-existing checkpoint (Required for Sample Selection)

In [ ]:
from google.colab import files
import os

os.makedirs("checkpoints", exist_ok=True)

# Upload your local resnet_pretrained_best.pth
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join("checkpoints", fname)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"Saved checkpoint → {dest}")

## Verify Model Access

In [ ]:
# Quick sanity check: load processor only (no GPU memory needed)
from transformers import AutoProcessor

model_id = "google/medgemma-4b-it"
print(f"Loading processor for {model_id} ...")
processor = AutoProcessor.from_pretrained(model_id)
print("✅ Processor loaded successfully!")
print(f"   Tokenizer class : {type(processor.tokenizer).__name__}")
print(f"   Image processor : {type(processor.image_processor).__name__}")
del processor  # free memory before loading full model

## Run Task 2 (Full Pipeline)
> This cell runs the complete Task 2 pipeline:
> 1. Selects 10 diverse test samples (Normal / Pneumonia / Misclassified)
> 2. Loads `google/medgemma-4b-it` in bfloat16 on GPU
> 3. Generates reports using **3 prompting strategies** per sample
> 4. Evaluates reports vs. ground truth and CNN predictions
> 5. Saves a Markdown report to `reports/task2_hf_medgemma_report.md`

In [ ]:
!python -m task2_report_generation.run_task2_hf \
    --model_id google/medgemma-4b-it \
    --num_samples 10 \
    --image_size 512 \
    --temperature 0.3 \
    --max_new_tokens 1024 \
    --cnn_model resnet \
    --cnn_pretrained true \
    --seed 42

## Run a Quick Single-Image Demo

In [ ]:
"""
Quick inline demo: load MedGemma and generate ONE report without the CLI.
Useful for debugging or interactive exploration.
"""
import sys
sys.path.insert(0, ".")

import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

# ── Load model ───────────────────────────────────────────────────────────────
model_id = "google/medgemma-4b-it"
device   = "cuda" if torch.cuda.is_available() else "cpu"
dtype    = torch.bfloat16 if device == "cuda" else torch.float32

print(f"Loading {model_id} on {device.upper()} ({dtype})…")
processor = AutoProcessor.from_pretrained(model_id)
model     = AutoModelForImageTextToText.from_pretrained(
    model_id, torch_dtype=dtype, device_map="auto"
)
model.eval()
print("✅ Model ready!")

# ── Create a synthetic test image (or load a real one) ───────────────────────
# Here we use a random grayscale image upscaled to 512x512 for demonstration.
# In the real pipeline, this comes from PneumoniaMNIST.
rng      = np.random.default_rng(42)
fake_img = (rng.random((28, 28)) * 255).astype(np.uint8)
pil_img  = Image.fromarray(fake_img, mode="L").resize((512, 512)).convert("RGB")

# ── Build prompt ─────────────────────────────────────────────────────────────
prompt_text = (
    "You are an expert radiologist. Analyze this chest X-ray image and provide "
    "a structured radiology report with the following sections:\n"
    "**EXAMINATION**: [modality and view]\n"
    "**FINDINGS**: [describe key observations]\n"
    "**IMPRESSION**: [Normal / Pneumonia with confidence]\n"
    "**CONFIDENCE**: [High / Medium / Low]"
)
messages = [{
    "role"   : "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": prompt_text},
    ],
}]

text_input = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(text=text_input, images=[pil_img], return_tensors="pt").to(model.device)

# ── Generate ──────────────────────────────────────────────────────────────────
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.3,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

n_input = inputs["input_ids"].shape[1]
report  = processor.decode(output_ids[0, n_input:], skip_special_tokens=True).strip()

print("\n" + "=" * 60)
print("GENERATED REPORT")
print("=" * 60)
print(report)

## Visualize Prompting Strategy Comparison

In [ ]:
import json, os
import matplotlib.pyplot as plt
import pandas as pd

strategy_path = "results/strategy_analysis.json"
if os.path.exists(strategy_path):
    with open(strategy_path) as f:
        strategy_data = json.load(f)

    strategies, agreements, lengths = [], [], []
    for name, stats in strategy_data.items():
        strategies.append(name)
        agreements.append(stats.get("gt_agreement_rate", 0) * 100)
        lengths.append(stats.get("avg_response_length", 0))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Task 2 — Prompting Strategy Comparison", fontsize=14, fontweight="bold")

    # GT Agreement
    colors = ["#4CAF50" if a >= 60 else "#FF9800" for a in agreements]
    axes[0].bar(strategies, agreements, color=colors, alpha=0.85, edgecolor="white")
    axes[0].set_ylabel("GT Agreement Rate (%)", fontsize=11)
    axes[0].set_title("Ground-Truth Agreement per Strategy", fontsize=12)
    axes[0].set_ylim(0, 100)
    for i, v in enumerate(agreements):
        axes[0].text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=10)

    # Response length
    axes[1].bar(strategies, lengths, color="#2196F3", alpha=0.85, edgecolor="white")
    axes[1].set_ylabel("Avg Response Length (tokens)", fontsize=11)
    axes[1].set_title("Average Report Length per Strategy", fontsize=12)
    for i, v in enumerate(lengths):
        axes[1].text(i, v + 5, str(int(v)), ha="center", fontsize=10)

    plt.tight_layout()
    plt.savefig("reports/task2_strategy_comparison.png", dpi=120)
    plt.show()
else:
    print("⚠️  Run Cell 6 first to generate strategy_analysis.json")

## Read the Generated Markdown Report

In [ ]:
report_path = "reports/task2_hf_medgemma_report.md"
if os.path.exists(report_path):
    with open(report_path) as f:
        content = f.read()
    # Display first 3000 characters
    print(content[:3000])
    print("\n... [truncated — open file for full report]")
else:
    print("⚠️  Report not found. Run Cell 6 to generate it.")